<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 20</h2>
<h2>Diffie-Hellman və ElGamal</h2>
<h3 style="color: #8B4513;">Diskret loqarifm probleminə əsaslanan kriptosistemlər</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: Asimmetrik kriptoqrafiya (5 dəq)](#xatırlatma-asimmetrik-kriptoqrafiya-5-dəq)
5. [Diffie–Hellman açar razılaşdırması (20 dəq)](#diffie-hellman-açar-mübadiləsi-20-dəq)
6. [ElGamal kriptosistemi (25 dəq)](#elgamal-kriptosistemi-25-dəq)
7. [Diskret loqarifm problemi (10 dəq)](#diskret-loqarifm-problemi-dlp-10-dəq)
8. [İnteqrasiya edilmiş özünü-yoxlama](#inteqrasiya-edilmiş-özünü-yoxlama-15-dəq)
9. [Ev tapşırığı](#ev-tapşırığı)
10. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
11. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ sonlu sahədə seçilmiş sadə tərtibli altqrupda Diffie–Hellman hesablayacaqsınız  
✅ peer public value validation, KDF və autentifikasiyanın rolunu izah edəcəksiniz  
✅ autentifikasiyasız DH üzərində MITM hücumunu göstərəcəksiniz  
✅ group-element ElGamal və hibrid ElGamal KEM-DEM modelini fərqləndirəcəksiniz  
✅ DLP, CDH və DDH fərziyyələrini və ümumi hücumların ölçüsünü müqayisə edəcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | İstifadə sahəsi |
|-----------|-----------------|
| Python standard library | Təhlükəsiz exponent/nonce seçimi, riyazi yoxlamalar |
| PyCryptodome | HKDF-SHA-256 və AES-GCM hybrid payload |

> 💡 **Mühit müqaviləsi:** notebook şəbəkədən paket quraşdırmır. `pycryptodome`
> cari kernel-də əvvəlcədən olmalıdır; import alınmasa run dərhal dayanır.
> `cryptography` və `gmpy2` istifadə edilmir.

In [ ]:
import importlib.util
import sys

missing = [name for module, name in {"Crypto": "pycryptodome"}.items()
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(
        "Çatışmayan paket(lər): " + ", ".join(missing)
        + ". Kernel mühitini notebook-dan əvvəl hazırlayın."
    )
print(f"✅ Mühit hazırdır: Python {sys.version.split()[0]}, pycryptodome mövcuddur")

## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import math
import secrets
import time
from pathlib import Path

from Crypto.Cipher import AES
from Crypto.Hash import SHA256
from Crypto.Protocol.KDF import HKDF
from Crypto.Random import get_random_bytes
from Crypto.Util.number import isPrime, long_to_bytes

# RFC 3526 MODP group 14: 2048-bit safe prime, generator 2.
MODP_2048_P = int(
    "FFFFFFFFFFFFFFFFC90FDAA22168C234C4C6628B80DC1CD1"
    "29024E088A67CC74020BBEA63B139B22514A08798E3404DD"
    "EF9519B3CD3A431B302B0A6DF25F14374FE1356D6D51C245"
    "E485B576625E7EC6F44C42E9A637ED6B0BFF5CB6F406B7ED"
    "EE386BFB5A899FA5AE9F24117C4B1FE649286651ECE45B3D"
    "C2007CB8A163BF0598DA48361C55D39A69163FA8FD24CF5F"
    "83655D23DCA3AD961C62F356208552BB9ED529077096966D"
    "670C354E4ABC9804F1746C08CA18217C32905E462E36CE3B"
    "E39E772C180E86039B2783A2EC07A28FB5C55DF06F4C52C9"
    "DE2BCBF6955817183995497CEA956AE515D2261898FA0510"
    "15728E5A8AACAA68FFFFFFFFFFFFFFFF",
    16,
)
MODP_2048_Q = (MODP_2048_P - 1) // 2
MODP_2048_G = 2
MODP_2048_BYTES = (MODP_2048_P.bit_length() + 7) // 8
print("✅ Standard library + PyCryptodome import edildi")

### 3.2 İşçi qovluğun yaradılması

In [ ]:
workspace = Path.cwd()
print(f"📁 İşçi qovluq dəyişdirilmir: {workspace}")

## 📖 Xatırlatma: Asimmetrik kriptoqrafiya (5 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 DLP əsaslı primitivlər</h4>
<ul>
    <li><b>DH:</b> autentifikasiya olunmuş protokol daxilində shared secret razılaşdırır; raw secret hazır session key deyil.</li>
    <li><b>ElGamal encryption:</b> seçilmiş qrupda public-key encryption modelidir; signature ayrıca sxemdir.</li>
    <li><b>Qrup:</b> modulun bit ölçüsü ilə yanaşı altqrupun sadə tərtibi, generator və peer public value validation vacibdir.</li>
</ul>
</div>

## 🔑 Diffie-Hellman açar mübadiləsi (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 Sonlu sahədə Diffie–Hellman</h4>
<p>Bu praktikada $p=2q+1$ safe prime, $q$ sadə və $g$ tərtibi $q$ olan altqrup generatorudur.</p>
<ol>
    <li>Alis $a\in\{2,\ldots,q-1\}$ seçib $A=g^a\bmod p$ göndərir.</li>
    <li>Bob $b\in\{2,\ldots,q-1\}$ seçib $B=g^b\bmod p$ göndərir.</li>
    <li>Hər tərəf peer value üçün range/subgroup validation edir.</li>
    <li>$s=B^a=A^b=g^{ab}\bmod p$ hesablanır.</li>
    <li>Session key raw $s$ deyil: eyni transcript/context ilə KDF-dən çıxarılır.</li>
</ol>
<p>Autentifikasiya olmadan aktiv MITM yenə mümkündür; qrup yoxlaması autentifikasiyanı əvəz etmir.</p>
</div>

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800; margin-top: 10px;">
<h4>📜 Tarixi qeyd</h4>
<p>Diffie və Hellman public-key cryptography və key agreement ideyalarını 1976-cı il “New Directions in Cryptography” məqaləsində açıq nəşr etdilər; Malcolm Williamson-un GCHQ daxilindəki əvvəlki işi sonradan deklassifikasiya edildi.</p>
</div>

In [ ]:
def validate_group(p, q, g):
    if p != 2 * q + 1 or not isPrime(p) or not isPrime(q):
        raise ValueError("p=2q+1 safe-prime qrupu tələb olunur")
    if not 1 < g < p - 1 or pow(g, q, p) != 1 or g == 1:
        raise ValueError("g tərtibi q olan qeyri-trivial element olmalıdır")

def validate_public(value, p, q):
    if not 1 < value < p - 1 or pow(value, q, p) != 1:
        raise ValueError("peer public value düzgün q-altqrupunda deyil")

def dh_exchange(p, q, g, alice_private, bob_private):
    validate_group(p, q, g)
    if not (2 <= alice_private < q and 2 <= bob_private < q):
        raise ValueError("private exponent 2..q-1 intervalında olmalıdır")
    alice_public = pow(g, alice_private, p)
    bob_public = pow(g, bob_private, p)
    validate_public(alice_public, p, q)
    validate_public(bob_public, p, q)
    alice_shared = pow(bob_public, alice_private, p)
    bob_shared = pow(alice_public, bob_private, p)
    assert alice_shared == bob_shared
    return alice_public, bob_public, alice_shared

toy_dh = dh_exchange(23, 11, 2, 6, 7)
assert toy_dh == (18, 13, 6)
print(f"✅ Toy DH: A={toy_dh[0]}, B={toy_dh[1]}, shared={toy_dh[2]}")

### 5.1 Named 2048-bit MODP qrupunda DH + KDF

In [ ]:
def dh_keypair(p=MODP_2048_P, q=MODP_2048_Q, g=MODP_2048_G):
    private = secrets.randbelow(q - 2) + 2
    public = pow(g, private, p)
    validate_public(public, p, q)
    return private, public

def dh_shared(private, peer_public, p=MODP_2048_P, q=MODP_2048_Q):
    validate_public(peer_public, p, q)
    shared = pow(peer_public, private, p)
    if shared in (0, 1):
        raise ValueError("degenerate shared secret")
    return shared

def derive_session_key(shared, context):
    return HKDF(
        long_to_bytes(shared, MODP_2048_BYTES), 32, b"", SHA256,
        context=context,
    )

def diffie_hellman_large():
    assert MODP_2048_P.bit_length() == 2048
    assert MODP_2048_P == 2 * MODP_2048_Q + 1
    assert pow(MODP_2048_G, MODP_2048_Q, MODP_2048_P) == 1
    alice_private, alice_public = dh_keypair()
    bob_private, bob_public = dh_keypair()
    alice_shared = dh_shared(alice_private, bob_public)
    bob_shared = dh_shared(bob_private, alice_public)
    assert alice_shared == bob_shared
    transcript = (
        b"AS20-DH-v1|"
        + long_to_bytes(alice_public, MODP_2048_BYTES)
        + long_to_bytes(bob_public, MODP_2048_BYTES)
    )
    alice_key = derive_session_key(alice_shared, transcript)
    bob_key = derive_session_key(bob_shared, transcript)
    assert alice_key == bob_key and len(alice_key) == 32
    print("✅ MODP-2048 DH: public validation və HKDF-derived 256-bit key keçdi")
    return alice_public, bob_public, alice_key

large_alice_public, large_bob_public, large_session_key = diffie_hellman_large()

### 5.2 Man-in-the-Middle hücumu

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ Man-in-the-Middle (MITM) Hücumu</h4>
<p>Diffie-Hellman protokolu autentifikasiya olmadan MITM hücumuna qarşı həssasdır.</p>
</div>

In [ ]:
def mitm_attack_demo():
    p, q, g = 23, 11, 2
    alice_private, bob_private = 6, 7
    alice_public = pow(g, alice_private, p)
    bob_public = pow(g, bob_private, p)

    mallory_to_alice_private = 3
    mallory_to_bob_private = 4
    mallory_to_alice_public = pow(g, mallory_to_alice_private, p)
    mallory_to_bob_public = pow(g, mallory_to_bob_private, p)

    alice_mallory = pow(mallory_to_alice_public, alice_private, p)
    mallory_alice = pow(alice_public, mallory_to_alice_private, p)
    bob_mallory = pow(mallory_to_bob_public, bob_private, p)
    mallory_bob = pow(bob_public, mallory_to_bob_private, p)
    assert alice_mallory == mallory_alice
    assert bob_mallory == mallory_bob
    assert alice_mallory != bob_mallory
    print(f"✅ MITM modeli: Alice↔Mallory={alice_mallory}, Bob↔Mallory={bob_mallory}")
    return alice_mallory, bob_mallory

mitm_keys = mitm_attack_demo()

### 5.3 Autentifikasiya edilmiş Diffie-Hellman

<div style="background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-left: 5px solid #4caf50;">
<h4>✅ Autentifikasiya edilmiş DH</h4>
<p>MITM-dən qorunmaq üçün yalnız $A$ və $B$ rəqəmlərini ayrıca imzalamaq kifayət deyil. Protokol rolları, identifikatorlar, qrup parametrləri və hər iki ephemeral public value vahid transcript-ə bağlanmalıdır; sertifikat/PSK yoxlanmalı, KDF context həmin transcript-i əhatə etməlidir. Hazır TLS/IKE/Noise protokolundan istifadə edin.</p>
</div>

### ✍️ Çalışma 1: Diffie–Hellman (1.5 bal)

1. $p=23,q=11,g=2,a=6,b=7$ üçün public values və shared secret-i hesablayın; hər public value-nun q-altqrupunda olduğunu yoxlayın.
2. $p=47,q=23,g=2,a=8,b=12$ üçün eyni hesablamanı aparın.
3. Niyə yalnız böyük $p$ demək kifayət deyil? $q$, $g$ və peer public validation rolunu izah edin.
4. Raw shared secret-dən session key almaq və MITM-dən qorunmaq üçün hansı ayrıca addımlar lazımdır?

In [ ]:
# Çalışma 1 - yoxlanılan cavablar
A1, B1, s1 = dh_exchange(23, 11, 2, 6, 7)
assert (A1, B1, s1) == (18, 13, 6)
A2, B2, s2 = dh_exchange(47, 23, 2, 8, 12)
assert pow(A2, 23, 47) == pow(B2, 23, 47) == 1
assert s2 == pow(B2, 8, 47) == pow(A2, 12, 47)
print(f"1. p=23: A={A1}, B={B1}, s={s1}")
print(f"2. p=47: A={A2}, B={B2}, s={s2}")
print("""
3. Təhlükəsizlik p bitindən əlavə böyük sadə q-tərtibli altqrupa, düzgün generatora,
   private exponent paylanmasına və peer public value validation-a bağlıdır.
4. Shared secret KDF + transcript/context ilə key-ə çevrilir; MITM üçün tərəflərin
   identikliyi və bütün handshake transcript-i imza/PSK/certificate ilə doğrulanır.
""")

## 🔐 ElGamal kriptosistemi (25 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔐 ElGamal encryption</h4>
<p>$G=\langle g\rangle$ sadə $q$ tərtibli qrup olsun.</p>
<ul>
    <li><b>Keygen:</b> $x\leftarrow\{2,\ldots,q-1\}$, $y=g^x$.</li>
    <li><b>Group-element encryption:</b> fresh uniform $k$; $c_1=g^k$, $c_2=m\,y^k$.</li>
    <li><b>Decryption:</b> $m=c_2(c_1^x)^{-1}$.</li>
    <li><b>Security model:</b> textbook ElGamal-ın IND-CPA iddiası seçilmiş qrupda DDH fərziyyəsi, düzgün message encoding və hər dəfə fresh uniform k tələb edir.</li>
    <li><b>Bytes:</b> arbitrary mətn group element deyil; praktik demo ElGamal-DH shared secret + HKDF + AES-GCM KEM-DEM istifadə edir.</li>
</ul>
</div>

In [ ]:
def egcd(a, b):
    old_r, r = a, b
    old_s, s = 1, 0
    old_t, t = 0, 1
    while r:
        quotient = old_r // r
        old_r, r = r, old_r - quotient * r
        old_s, s = s, old_s - quotient * s
        old_t, t = t, old_t - quotient * t
    return abs(old_r), old_s, old_t

def modinv(a, modulus):
    if modulus <= 1:
        raise ValueError("modulus > 1 olmalıdır")
    gcd_value, x, _ = egcd(a % modulus, modulus)
    if gcd_value != 1:
        raise ValueError("modul tərs mövcud deyil")
    result = x % modulus
    assert (a * result) % modulus == 1
    return result

In [ ]:
def elgamal_keygen(p=MODP_2048_P, q=MODP_2048_Q, g=MODP_2048_G):
    validate_group(p, q, g)
    private = secrets.randbelow(q - 2) + 2
    public_value = pow(g, private, p)
    validate_public(public_value, p, q)
    return (p, q, g, public_value), private

elgamal_public_key, elgamal_private_key = elgamal_keygen()
print("✅ ElGamal keygen: 2048-bit named group, private exponent ekrana çıxarılmır")

In [ ]:
def validate_group_element(value, p, q):
    if not 1 <= value < p or pow(value, q, p) != 1:
        raise ValueError("dəyər seçilmiş q-altqrupunda deyil")

def elgamal_encrypt_element(message, public_key, nonce=None):
    p, q, g, public_value = public_key
    validate_group(p, q, g)
    validate_public(public_value, p, q)
    validate_group_element(message, p, q)
    k = secrets.randbelow(q - 2) + 2 if nonce is None else nonce
    if not 2 <= k < q:
        raise ValueError("nonce 2..q-1 intervalında olmalıdır")
    c1 = pow(g, k, p)
    shared = pow(public_value, k, p)
    c2 = (message * shared) % p
    validate_public(c1, p, q)
    validate_group_element(c2, p, q)
    return c1, c2

def elgamal_decrypt_element(ciphertext, private_key, public_key):
    p, q, g, public_value = public_key
    c1, c2 = ciphertext
    validate_public(c1, p, q)
    validate_group_element(c2, p, q)
    shared = pow(c1, private_key, p)
    message = (c2 * modinv(shared, p)) % p
    validate_group_element(message, p, q)
    return message

def elgamal_demo():
    public_key, private_key = elgamal_keygen()
    p, q, g, _ = public_key
    message = pow(g, 42, p)
    ciphertext = elgamal_encrypt_element(message, public_key)
    recovered = elgamal_decrypt_element(ciphertext, private_key, public_key)
    assert recovered == message
    print("✅ Group-element ElGamal round-trip keçdi")
    return public_key, private_key, message, ciphertext

demo_public, demo_private, demo_message, demo_ciphertext = elgamal_demo()

### 6.1 ElGamal-ın ehtimalli xarakteri

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🎲 Fresh nonce və təhlükəsizlik fərziyyəsi</h4>
<p>Fresh uniform $k$ eyni group element üçün adətən fərqli ciphertext yaradır və nonce reuse-i qadağandır. Lakin “fərqli ciphertext” müşahidəsi təkbaşına IND-CPA sübutu deyil: textbook ElGamal üçün DDH-hard group və düzgün message space/encoding də tələb olunur.</p>
</div>

In [ ]:
def elgamal_probabilistic_demo():
    public_key, private_key = elgamal_keygen()
    p, q, g, _ = public_key
    message = pow(g, 77, p)
    ciphertexts = [elgamal_encrypt_element(message, public_key) for _ in range(5)]
    recovered = [elgamal_decrypt_element(c, private_key, public_key) for c in ciphertexts]
    assert all(value == message for value in recovered)
    assert len(set(ciphertexts)) > 1
    print(f"✅ 5 encryption round-trip; distinct ciphertext sayı={len(set(ciphertexts))}")
    return ciphertexts

probabilistic_ciphertexts = elgamal_probabilistic_demo()

### 6.2 Mətn üçün hibrid ElGamal KEM-DEM

In [ ]:
def elgamal_kdf(shared, c1, recipient_public):
    context = (
        b"AS20-ElGamal-KEM-v1|"
        + long_to_bytes(c1, MODP_2048_BYTES)
        + long_to_bytes(recipient_public, MODP_2048_BYTES)
    )
    return derive_session_key(shared, context)

def elgamal_hybrid_encrypt(message, public_key):
    p, q, g, recipient_public = public_key
    k = secrets.randbelow(q - 2) + 2
    c1 = pow(g, k, p)
    shared = pow(recipient_public, k, p)
    key = elgamal_kdf(shared, c1, recipient_public)
    cipher = AES.new(key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(message)
    return c1, cipher.nonce, ciphertext, tag

def elgamal_hybrid_decrypt(package, private_key, public_key):
    p, q, g, recipient_public = public_key
    c1, nonce, ciphertext, tag = package
    validate_public(c1, p, q)
    shared = pow(c1, private_key, p)
    key = elgamal_kdf(shared, c1, recipient_public)
    return AES.new(key, AES.MODE_GCM, nonce=nonce).decrypt_and_verify(ciphertext, tag)

hybrid_message = "Salam, ElGamal KEM-DEM!".encode("utf-8")
hybrid_package = elgamal_hybrid_encrypt(hybrid_message, elgamal_public_key)
assert elgamal_hybrid_decrypt(
    hybrid_package, elgamal_private_key, elgamal_public_key
) == hybrid_message
print(f"✅ Hybrid ElGamal: {len(hybrid_message)} bayt round-trip və AES-GCM integrity")

### ✍️ Çalışma 2: ElGamal (1.5 bal)

1. $p=23,q=11,g=2,x=6$ üçün $y$-ni hesablayın və subgroup membership-i yoxlayın.
2. Group element $m=g^4\bmod p$-ni tədris nonce-u $k=3$ ilə şifrləyin.
3. Ciphertext-i deşifrləyib $m$ ilə assertion qurun.
4. Fresh uniform $k$, qrup/DDH fərziyyəsi və arbitrary bytes üçün KEM-DEM ehtiyacını ayrı izah edin.

In [ ]:
# Çalışma 2 - yoxlanılan cavablar
p2, q2, g2, x2 = 23, 11, 2, 6
y2 = pow(g2, x2, p2)
public2 = (p2, q2, g2, y2)
message2 = pow(g2, 4, p2)
ciphertext2 = elgamal_encrypt_element(message2, public2, nonce=3)
recovered2 = elgamal_decrypt_element(ciphertext2, x2, public2)
assert y2 == 18 and pow(y2, q2, p2) == 1
assert message2 == 16 and ciphertext2 == (8, 1)
assert recovered2 == message2
print(f"1–3. y={y2}, m={message2}, ciphertext={ciphertext2}, bərpa={recovered2}")
print("""
4. Fresh uniform k nonce reuse-i və deterministik ciphertext-i önləyir, amma tək
   şərt deyil. Textbook ElGamal IND-CPA iddiası DDH-hard qrup və düzgün message
   encoding tələb edir; arbitrary bytes üçün authenticated KEM-DEM istifadə olunur.
""")

## 🔢 Diskret loqarifm problemi (DLP) (10 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔢 DLP, CDH və DDH</h4>
<ul>
    <li><b>DLP:</b> $g,y=g^x$ verildikdə $x$-i tapmaq.</li>
    <li><b>CDH:</b> $g^a,g^b$ verildikdə $g^{ab}$ hesablamaq.</li>
    <li><b>DDH:</b> $(g^a,g^b,g^c)$ üçün $c=ab$ olub-olmadığını ayırmaq.</li>
</ul>
<p>DH/ElGamal təhlükəsizlik iddiaları konkret qrupda uyğun CDH/DDH fərziyyəsinə və tam protokola bağlıdır; yalnız “DLP çətindir” cümləsi bütün iddiaları sübut etmir.</p>
</div>

In [ ]:
def discrete_log_bruteforce(g, target, p, order):
    value = 1
    for exponent in range(order):
        if value == target:
            return exponent
        value = (value * g) % p
    raise ValueError("target seçilmiş altqrupda tapılmadı")

toy_target = pow(2, 7, 23)
toy_log = discrete_log_bruteforce(2, toy_target, 23, 11)
assert toy_target == 13 and toy_log == 7
print(f"✅ Toy DLP: 2^x ≡ {toy_target} (mod 23), x={toy_log}")

### 7.1 DLP hücum üsulları

In [ ]:
def baby_step_giant_step(g, target, p, order):
    m = math.isqrt(order) + 1
    table = {}
    value = 1
    for j in range(m):
        table.setdefault(value, j)
        value = (value * g) % p
    factor = modinv(pow(g, m, p), p)
    gamma = target
    for i in range(m + 1):
        if gamma in table:
            candidate = i * m + table[gamma]
            if candidate < order and pow(g, candidate, p) == target:
                return candidate
        gamma = (gamma * factor) % p
    raise ValueError("diskret loqarifm tapılmadı")

assert baby_step_giant_step(2, 13, 23, 11) == 7
print("""
Hücum konteksti:
• brute force: O(q) group operations;
• baby-step giant-step: O(√q) time və O(√q) memory;
• Pollard rho: expected O(√q) time, kiçik memory;
• Pohlig–Hellman: q smooth olduqda;
• finite-field index calculus/NFS-DL: p ölçüsündən asılı subexponential hücum.

Ölçü yalnız “p böyükdür” ilə seçilmir; standart named group və tələb olunan
security strength/cryptoperiod istifadə olunur. 2048-bit MODP təxminən 112-bit
classical strength səviyyəsidir, universal “həmişə təhlükəsiz” hökmü deyil.
""")

### ✍️ Çalışma 3: Diskret loqarifm (1 bal)

1. $p=23,q=11,g=2,y=13$ üçün DLP-ni brute force və BSGS ilə tapın.
2. $p=47,q=23,g=2,y=pow(2,9,47)$ üçün hər iki üsulu müqayisə edin.
3. Generic hücumlarda complexity-ni niyə $p$ yox, subgroup order $q$ ilə yazırıq?
4. Finite-field DLP üçün index calculus/NFS-DL group-size seçiminə necə təsir edir?

In [ ]:
# Çalışma 3 - yoxlanılan cavablar
cases = ((23, 11, 2, 7), (47, 23, 2, 9))
for p3, q3, g3, expected in cases:
    target3 = pow(g3, expected, p3)
    brute = discrete_log_bruteforce(g3, target3, p3, q3)
    bsgs = baby_step_giant_step(g3, target3, p3, q3)
    assert brute == bsgs == expected
    print(f"p={p3}, q={q3}, y={target3}: x={expected}")

print("""
Generic algorithms q elementli subgroup üzərində işləyir: exhaustive O(q),
BSGS/Pollard rho təxminən O(√q). Finite fields üçün NFS-DL p-yə görə
subexponential olduğundan həm p, həm q və standart qrup seçimi vacibdir.
DLP/CDH/DDH əlaqəli, lakin eyni problem deyil; security claim sxemə görə seçilir.
""")

## 🖥️ İnteqrasiya edilmiş özünü-yoxlama (15 dəq)

In [ ]:
def dh_elgamal_self_test():
    assert dh_exchange(23, 11, 2, 6, 7) == (18, 13, 6)
    assert mitm_attack_demo() == mitm_keys
    assert elgamal_decrypt_element(
        demo_ciphertext, demo_private, demo_public
    ) == demo_message
    assert elgamal_hybrid_decrypt(
        hybrid_package, elgamal_private_key, elgamal_public_key
    ) == hybrid_message
    assert baby_step_giant_step(2, 13, 23, 11) == 7
    return True

assert dh_elgamal_self_test()
print("✅ DH/ElGamal integrated self-test tamamlandı")

## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: DH və ElGamal tədris paketi (3 bal)

```text
dh_elgamal_package/
├── groups.py       # named parameters və validation
├── dh.py           # DH shared secret + KDF context
├── elgamal.py      # group-element model
├── hybrid.py       # ElGamal KEM + HKDF + AEAD
├── dlp.py          # brute force və BSGS
└── tests/          # positive/negative/property tests
```

Private exponent/nonce üçün `secrets` istifadə edin; peer values və domains-i
yoxlayın; raw shared secret-i key adlandırmayın; textbook hissəni production API
kimi təqdim etməyin; invalid subgroup, nonce reuse və tamper üçün mənfi test yazın.

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

1. **Named-group DH:** RFC 3526 group 14 və ya protokolun tələb etdiyi named group ilə peer validation + transcript-bound KDF qurun.
2. **Hybrid ElGamal:** faylı AEAD ilə şifrləyin, shared secret-dən HKDF key çıxarın; tag/c1 dəyişikliklərini rədd edin.
3. **Nonce testi:** eyni group element-i fresh secure nonce ilə 10 dəfə şifrləyin və hamısını deşifrləyin; fərqliliyi security proof adlandırmayın.
4. **BSGS:** kiçik qruplarda bütün subgroup elementləri üçün DLP-ni bərpa edən property test yazın.

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

1. Diffie–Hellman (1976) məqaləsində public-key distribution və authentication sərhədi necə qoyulur?
2. ElGamal (1985) encryption və signature constructions necə ayrılır?
3. NIST SP 800-56A Rev. 3 finite-field public-key validation və key derivation üçün nə tələb edir?
4. Generic DLP attacks ilə finite-field NFS-DL-ni, sonra Şor alqoritminin threat modelini müqayisə edin.

Hesabat tarixini göstərin; ən azı 5 mənbədən RFC/NIST/orijinal məqalələrə üstünlük verin və gələcək “təhlükəsiz qalacaq il” proqnozu verməyin.

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<ul>
    <li>✅ <b>Qrup:</b> safe prime $p=2q+1$, sadə q-tərtibli altqrup və public value validation.</li>
    <li>✅ <b>DH:</b> raw $g^{ab}$ deyil, transcript-bound KDF nəticəsi session key-dir.</li>
    <li>✅ <b>MITM:</b> qrup yoxlaması vacibdir, lakin identity/transcript authentication ayrıca tələb olunur.</li>
    <li>✅ <b>ElGamal:</b> group-element sxemi DDH/fresh nonce şərtləri ilə; bytes üçün authenticated KEM-DEM.</li>
    <li>✅ <b>DLP/CDH/DDH:</b> fərqli problemlərdir; generic complexity subgroup order q ilə ölçülür, finite-field index calculus p-ni də vacib edir.</li>
</ul>
</div>

### 💭 Müzakirə sualları

1. Peer public value niyə yoxlanmalıdır?
2. Raw DH shared secret niyə birbaşa tətbiq açarı deyil?
3. Autentifikasiya hansı transcript elementlərini bağlamalıdır?
4. ElGamal randomization hansı şərtlərlə IND-CPA iddiasına çevrilir?
5. Generic √q hücumu ilə finite-field NFS-DL arasında fərq nədir?

## 📚 Əlavə resurslar

* **Diffie & Hellman (1976):** [New Directions in Cryptography](https://ee.stanford.edu/~hellman/publications/24.pdf)
* **ElGamal (1985):** [A Public Key Cryptosystem and a Signature Scheme Based on Discrete Logarithms](https://doi.org/10.1109/TIT.1985.1057074)
* **NIST SP 800-56A Rev. 3:** [Discrete-log key establishment](https://csrc.nist.gov/pubs/sp/800/56/a/r3/final)
* **RFC 3526:** [MODP groups for IKE](https://www.rfc-editor.org/info/rfc3526/)
* **RFC 7919:** [Named FFDHE groups and public-key checks for TLS](https://www.rfc-editor.org/info/rfc7919/)
* **NIST SP 800-57 Part 1 Rev. 5:** [Security strength mapping](https://csrc.nist.gov/pubs/sp/800/57/pt1/r5/final)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 20 tamamlandı!</h2>
<p>Bütün code cells assertions ilə yuxarıdan aşağı işləməlidir.</p>
</div>